# TF-IDF Baseline
**Project:** Origin | **Status:** starter template — not yet implemented

## Objective
Fit the lexical vectorizer on project records and inspect Top-K retrieval.

**Expected output:** `results/tfidf_*.csv`

> Research decisions in the report are provisional. Do not claim evaluated performance before running experiments.


## 1. Setup and dependencies
Use Google Colab and authenticate to the repository securely. Each notebook must load its own inputs. Never print secrets.


In [7]:
%pip -q install pandas scikit-learn openpyxl

In [8]:
import io, json, re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

OUTPUT_DIR = Path('results')
OUTPUT_DIR.mkdir(exist_ok=True)
TOP_K = 5
RANDOM_STATE = 42
print('Ready. scikit-learn version loaded.')

Ready. scikit-learn version loaded.


## 2. Load inputs and verify prerequisites
Document file paths, data assumptions, and expected schema before running experiments.


In [9]:
# Load the validated processed dataset directly from GitHub

PROJECTS_URL = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/processed/projects_clean.csv"
)

df = pd.read_csv(PROJECTS_URL, encoding="utf-8-sig")

print("Dataset loaded successfully from GitHub.")
print("Dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())

display(df.head(3))

Dataset loaded successfully from GitHub.
Dataset shape: (103, 7)
Available columns: ['Project_ID', 'Title', 'Abstract', 'Department', 'Year', 'Keywords', 'Source_Page']


,Project_ID,Title,Abstract,Department,Year,Keywords,Source_Page
0,P001,Automatic Plant Watering and Tracking System,Many people struggle with watering their plant...,Computer Science,2024 – 2025,"Moisturem Air humidity, AWTS system, Waterin, ...",5
1,P002,علامة CourseAssist: CLO Performance Assistant ...,The rapid evolution of educational technology ...,Computer Science,2024 – 2025,"Program Learning Outcomes, Course Learning Out...",6
2,P003,واصف Mobile Application for Deaf and Mute indi...,"Our System, Wasif, offers a communication appl...",Computer Science,2024 – 2025,"Arabic Sign Language, Sign Language Gestures i...",7


In [10]:
# Adjust these names ONLY if auto-detection does not match your dataset.
OVERRIDE_ID = None
OVERRIDE_TITLE = None
OVERRIDE_DESCRIPTION = None
OVERRIDE_TEXT = None  # Use only when one column already contains complete project text

aliases = {
    'id': ['project_id', 'id', 'project number', 'project_number', 'رقم المشروع', 'معرف المشروع'],
    'title': ['title', 'project_title', 'project name', 'project_name', 'اسم المشروع', 'عنوان المشروع'],
    'description': ['description', 'abstract', 'project_description', 'summary', 'idea', 'الوصف', 'وصف المشروع', 'ملخص', 'الفكرة'],
    'text': ['combined_text', 'project_text', 'cleaned_text', 'text', 'full_text', 'search_text']
}

def detect_column(columns, candidates):
    lookup = {str(c).strip().casefold(): c for c in columns}
    return next((lookup[a.casefold()] for a in candidates if a.casefold() in lookup), None)

id_col = OVERRIDE_ID or detect_column(df.columns, aliases['id'])
title_col = OVERRIDE_TITLE or detect_column(df.columns, aliases['title'])
desc_col = OVERRIDE_DESCRIPTION or detect_column(df.columns, aliases['description'])
text_col = OVERRIDE_TEXT or detect_column(df.columns, aliases['text'])
print({'id': id_col, 'title': title_col, 'description': desc_col, 'text': text_col})
assert text_col is not None or title_col is not None or desc_col is not None, (
    'Could not identify text columns. Set OVERRIDE_TITLE / OVERRIDE_DESCRIPTION / OVERRIDE_TEXT above.'
)
assert df.shape[0] > 0, 'No rows in dataset.'

{'id': 'Project_ID', 'title': 'Title', 'description': 'Abstract', 'text': None}


In [11]:
def clean_text(value):
    # Minimal normalization only: keep Arabic and English letters and avoid aggressive stemming.
    if pd.isna(value):
        return ''
    txt = str(value).strip().lower()
    txt = re.sub(r'[\u064b-\u065f\u0670]', '', txt)  # Arabic diacritics
    txt = re.sub(r'\s+', ' ', txt)
    return txt

records = pd.DataFrame()
records['project_id'] = (df[id_col].astype(str) if id_col else pd.Series(
    [f'P{i:03d}' for i in range(1, len(df)+1)], index=df.index))
records['title'] = df[title_col].fillna('').astype(str) if title_col else ''
records['description'] = df[desc_col].fillna('').astype(str) if desc_col else ''
if text_col is not None:
    records['search_text'] = df[text_col].map(clean_text)
else:
    records['search_text'] = (records['title'] + ' ' + records['description']).map(clean_text)
records = records[records['search_text'].str.len() > 0].copy().reset_index(drop=True)
assert len(records) > 0, 'No nonempty searchable texts.'
assert records['project_id'].is_unique, 'Project IDs are not unique: check source data or ID selection.'
print('Valid project records:', len(records))
print('Empty records skipped:', len(df) - len(records))
print('Repeated text rows:', records['search_text'].duplicated().sum())
display(records[['project_id','title','search_text']].head(3))

Valid project records: 103
Empty records skipped: 0
Repeated text rows: 0


,project_id,title,search_text
0,P001,Automatic Plant Watering and Tracking System,automatic plant watering and tracking system m...
1,P002,علامة CourseAssist: CLO Performance Assistant ...,علامة courseassist: clo performance assistant ...
2,P003,واصف Mobile Application for Deaf and Mute indi...,واصف mobile application for deaf and mute indi...


## 3. Implementation / experiments
Build and test this step incrementally; document any methodological changes.


In [12]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    analyzer='word',
    ngram_range=(1, 2),
    min_df=1,
    max_df=1.0,
    sublinear_tf=True,
    token_pattern=r'(?u)\b\w+\b'
)
corpus_matrix = vectorizer.fit_transform(records['search_text'])
print('TF-IDF document-term matrix:', corpus_matrix.shape)
print('Vocabulary size:', len(vectorizer.vocabulary_))

TF-IDF document-term matrix: (103, 16881)
Vocabulary size: 16881


In [13]:
def retrieve(query, k=TOP_K, excluded_project_id=None):
    query = clean_text(query)
    if not query:
        raise ValueError('Query must not be empty.')
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, corpus_matrix).ravel()
    candidates = np.argsort(-sims, kind='stable')
    if excluded_project_id is not None:
        candidates = [i for i in candidates if records.iloc[i]['project_id'] != str(excluded_project_id)]
    selected = list(candidates)[:min(k, len(candidates))]
    ans = records.iloc[selected][['project_id','title','description']].copy()
    ans.insert(0, 'rank', range(1, len(ans)+1))
    ans['cosine_similarity'] = np.round(sims[selected], 6)
    return ans.reset_index(drop=True)

# Test with a NEW project idea rather than a corpus record.
TEST_QUERY = 'AI system for recognizing Arabic sign language and helping deaf people communicate'
result = retrieve(TEST_QUERY, k=TOP_K)
display(result)

,rank,project_id,title,description,cosine_similarity
0,1,P069,Bridging Communication Gaps: A system for Arab...,Sign language plays a critical role in communi...,0.175714
1,2,P003,واصف Mobile Application for Deaf and Mute indi...,"Our System, Wasif, offers a communication appl...",0.119429
2,3,P036,رَفد RafdSign: Audio-to-Sign Language Translator,Individuals with hearing impairments face sign...,0.109432
3,4,P047,ASMAAK | أسمعك,The “ASMAAK | أسمعك” project is a child-friend...,0.087267
4,5,P074,بصيرة Baseera: An AI-Based Sound Recognition a...,Individuals with hearing impairments often enc...,0.051667


## 4. Validate results
Run sanity checks, inspect failure cases, and avoid interpreting uncalibrated scores as duplication probabilities.


In [14]:
# Verify rankings, ranges, and query vocabulary coverage.
assert len(result) <= TOP_K
assert result['cosine_similarity'].between(0,1).all()
assert result['cosine_similarity'].is_monotonic_decreasing
assert result['project_id'].is_unique

query_vec = vectorizer.transform([clean_text(TEST_QUERY)])
query_known_terms = query_vec.nnz
print('TEST_QUERY:', TEST_QUERY)
print('Known query terms/ngrams:', query_known_terms)
print('Top similarity:', result['cosine_similarity'].iloc[0] if len(result) else 'N/A')
if query_known_terms == 0:
    print('WARNING: Query shares no known vocabulary with this corpus; all scores can be zero.')
print('Checks passed. Retrieval relevance NOT YET EVALUATED.')

TEST_QUERY: AI system for recognizing Arabic sign language and helping deaf people communicate
Known query terms/ngrams: 17
Top similarity: 0.175714
Checks passed. Retrieval relevance NOT YET EVALUATED.


## 5. Save outputs and conclusions
Record output paths, reproducibility notes, decisions, and next actions.


In [16]:
topk_export = result.copy()
topk_export.insert(0, 'query_id', 'Q_DEMO_001')
topk_export.insert(1, 'query_text', TEST_QUERY)

# Exclude potentially restricted descriptions from shareable outputs
topk_export = topk_export.drop(columns=['description'])

topk_export.to_csv(
    OUTPUT_DIR / 'tfidf_topk.csv',
    index=False,
    encoding='utf-8-sig'
)

pd.DataFrame([
    {
        'query_id': 'Q_DEMO_001',
        'query_text': TEST_QUERY,
        'type': 'demo_only_not_human_labeled'
    }
]).to_csv(
    OUTPUT_DIR / 'tfidf_queries.csv',
    index=False,
    encoding='utf-8-sig'
)

summary = {
    'model': 'TfidfVectorizer + cosine_similarity',
    'ngram_range': [1, 2],
    'min_df': 1,
    'max_df': 1.0,
    'sublinear_tf': True,
    'top_k': TOP_K,
    'source_file': PROJECTS_URL,
    'input_row_count': len(df),
    'valid_record_count': len(records),
    'vocabulary_size': len(vectorizer.vocabulary_),
    'evaluation_status': 'Not evaluated against human relevance judgments',
    'notes': 'Scores are cosine similarities, not probability of duplication.'
}

(OUTPUT_DIR / 'tfidf_run_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding='utf-8'
)

print('Saved files:')
for path in OUTPUT_DIR.glob('tfidf_*'):
    print(path, '|', path.stat().st_size, 'bytes')

Saved files:
results/tfidf_topk.csv | 915 bytes
results/tfidf_queries.csv | 150 bytes
results/tfidf_run_summary.json | 513 bytes


In [17]:
# Optional: Download the three result files in one ZIP from Colab.
import shutil
from google.colab import files
archive = shutil.make_archive('origin_tfidf_results', 'zip', root_dir='.', base_dir='results')
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#-------------------------------------------------------

## 6. Batch Retrieval on Development Queries

Use the development split of the evaluation dataset to test the TF-IDF baseline on multiple realistic project ideas.

Only the `dev` queries are used during development. The `final_test` queries remain reserved for the final model evaluation.## 5. Batch Retrieval on Development Queries

Use the development split of the evaluation dataset to test the TF-IDF baseline on multiple realistic project ideas.

Only the `dev` queries are used during development. The `final_test` queries remain reserved for the final model evaluation.

In [18]:
EVAL_URL = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/evaluation/evaluation_queries.csv"
)

eval_df = pd.read_csv(EVAL_URL, encoding="utf-8-sig")

print("Evaluation dataset shape:", eval_df.shape)
print("Split distribution:")
print(eval_df["split"].value_counts())

Evaluation dataset shape: (60, 7)
Split distribution:
split
dev           40
final_test    20
Name: count, dtype: int64


In [19]:
dev_queries = (
    eval_df[eval_df["split"] == "dev"]
    .copy()
    .reset_index(drop=True)
)

# Origin receives a project title + brief description
dev_queries["query_text"] = (
    dev_queries["query_title"].fillna("").str.strip()
    + " "
    + dev_queries["query_description"].fillna("").str.strip()
).str.strip()

print("Development queries:", len(dev_queries))

display(
    dev_queries[
        [
            "query_id",
            "query_title",
            "language",
            "generation_intent",
            "query_text"
        ]
    ].head()
)

Development queries: 40


,query_id,query_title,language,generation_intent,query_text
0,Q003,Blood Stock Planning Desk,en,near_overlap,Blood Stock Planning Desk Hospital blood-bank ...
1,Q004,Everyday Sound Notifier,en,near_overlap,Everyday Sound Notifier Deaf residents may mis...
2,Q006,مؤشرات تحقق المخرجات الدراسية,ar,near_overlap,مؤشرات تحقق المخرجات الدراسية نقترح منصة لأعضا...
3,Q010,منظم فرص التطوع وساعات المشاركة,ar,near_overlap,منظم فرص التطوع وساعات المشاركة تحتاج الجمعيات...
4,Q011,Plant Nursery Outbreak Map,en,related_distinct,Plant Nursery Outbreak Map Nursery managers ne...


In [20]:
# Run TF-IDF retrieval for all development queries

all_results = []

for _, row in dev_queries.iterrows():
    query_id = row["query_id"]
    query_text = row["query_text"]

    retrieved = retrieve(query_text, k=TOP_K).copy()

    retrieved.insert(0, "query_id", query_id)
    retrieved.insert(1, "query_text", query_text)

    all_results.append(retrieved)

tfidf_dev_results = pd.concat(all_results, ignore_index=True)

print("Total development queries:", len(dev_queries))
print("Top-K per query:", TOP_K)
print("Total retrieved rows:", len(tfidf_dev_results))

display(tfidf_dev_results.head(10))

Total development queries: 40
Top-K per query: 5
Total retrieved rows: 200


,query_id,query_text,rank,project_id,title,description,cosine_similarity
0,Q003,Blood Stock Planning Desk Hospital blood-bank ...,1,P059,Blood Bank Inventory System - Artificial Intel...,Healthcare institutions are usually challenged...,0.188804
1,Q003,Blood Stock Planning Desk Hospital blood-bank ...,2,P038,UniFix— Predictive Maintenance,Predictive maintenance is a modern approach to...,0.052368
2,Q003,Blood Stock Planning Desk Hospital blood-bank ...,3,P081,تنبا Artificial Intelligence for Predicting St...,This project proposes a stock price prediction...,0.051268
3,Q003,Blood Stock Planning Desk Hospital blood-bank ...,4,P070,Mama’s Hope,Preeclampsia is a life-threatening pregnancy c...,0.051181
4,Q003,Blood Stock Planning Desk Hospital blood-bank ...,5,P028,Easy Picks,In a world increasingly reliant on online shop...,0.046299
5,Q004,Everyday Sound Notifier Deaf residents may mis...,1,P074,بصيرة Baseera: An AI-Based Sound Recognition a...,Individuals with hearing impairments often enc...,0.096683
6,Q004,Everyday Sound Notifier Deaf residents may mis...,2,P008,SyncUp,Remote teamwork has revolutionized project imp...,0.062300
7,Q004,Everyday Sound Notifier Deaf residents may mis...,3,P057,PopList,This project aims to create a mobile applicati...,0.061527
8,Q004,Everyday Sound Notifier Deaf residents may mis...,4,P012,خلها علينا ‘Khalaha Alyna’ – Event Management App,"This project aims to develop Khalaha Alyna, a ...",0.053283
9,Q004,Everyday Sound Notifier Deaf residents may mis...,5,P001,Automatic Plant Watering and Tracking System,Many people struggle with watering their plant...,0.053224


In [21]:
# Validate batch retrieval results

print("Unique queries:", tfidf_dev_results["query_id"].nunique())

print("\nResults per query:")
print(tfidf_dev_results.groupby("query_id").size().value_counts())

print(
    "\nSimilarity score range:",
    tfidf_dev_results["cosine_similarity"].min(),
    "to",
    tfidf_dev_results["cosine_similarity"].max()
)

assert tfidf_dev_results["query_id"].nunique() == 40
assert len(tfidf_dev_results) == 40 * TOP_K
assert tfidf_dev_results["cosine_similarity"].between(0, 1).all()

print("\n✓ Batch retrieval checks passed.")

Unique queries: 40

Results per query:
5    40
Name: count, dtype: int64

Similarity score range: 0.0 to 0.188804

✓ Batch retrieval checks passed.


## 7. Development Query Diagnostics

Inspect TF-IDF vocabulary coverage and retrieval behavior across the development queries before saving the baseline results.

In [24]:
# Development query diagnostics for TF-IDF baseline

diagnostics = []

for _, row in dev_queries.iterrows():
    query_id = row["query_id"]
    query_text = row["query_text"]

    # Measure how much of the query exists in the TF-IDF vocabulary
    query_vec = vectorizer.transform([clean_text(query_text)])
    known_terms = query_vec.nnz

    # Get the Top-K retrieval results for this query
    query_results = tfidf_dev_results[
        tfidf_dev_results["query_id"] == query_id
    ]

    top_score = query_results["cosine_similarity"].max()
    zero_scores = (query_results["cosine_similarity"] == 0).sum()

    # Provenance references are used only for sanity checking.
    # They are NOT final human relevance ground truth.
    refs = row["reference_project_ids"]

    # If TF-IDF retrieved only zero scores, we cannot treat a reference
    # appearing in Top-K as a meaningful hit because the ranking is arbitrary.
    if top_score == 0:
        reference_hit_top5 = None

    # Distant-novel queries may intentionally have no reference project.
    elif pd.isna(refs):
        reference_hit_top5 = None

    else:
        reference_ids = {
            ref.strip()
            for ref in str(refs).split(";")
            if ref.strip()
        }

        retrieved_ids = set(query_results["project_id"])

        reference_hit_top5 = bool(
            reference_ids & retrieved_ids
        )

    diagnostics.append({
        "query_id": query_id,
        "language": row["language"],
        "generation_intent": row["generation_intent"],
        "known_terms_ngrams": known_terms,
        "top_cosine_similarity": top_score,
        "zero_scores_in_top5": zero_scores,
        "reference_hit_top5": reference_hit_top5
    })


tfidf_diagnostics = pd.DataFrame(diagnostics)

print("TF-IDF DEVELOPMENT DIAGNOSTICS")
print("=" * 60)

print(
    "Queries with zero known TF-IDF terms:",
    (tfidf_diagnostics["known_terms_ngrams"] == 0).sum()
)

print(
    "Queries whose Top-1 similarity is zero:",
    (tfidf_diagnostics["top_cosine_similarity"] == 0).sum()
)

print("\nAverage Top-1 similarity by language:")

print(
    tfidf_diagnostics
    .groupby("language")["top_cosine_similarity"]
    .mean()
)

print("\nDetailed diagnostics:")
display(tfidf_diagnostics)

TF-IDF DEVELOPMENT DIAGNOSTICS
Queries with zero known TF-IDF terms: 19
Queries whose Top-1 similarity is zero: 19

Average Top-1 similarity by language:
language
ar    0.003524
en    0.092851
Name: top_cosine_similarity, dtype: float64

Detailed diagnostics:


,query_id,language,generation_intent,known_terms_ngrams,top_cosine_similarity,zero_scores_in_top5,reference_hit_top5
0,Q003,en,near_overlap,59,0.188804,0,True
1,Q004,en,near_overlap,49,0.096683,0,True
2,Q006,ar,near_overlap,0,0.000000,5,None
3,Q010,ar,near_overlap,0,0.000000,5,None
4,Q011,en,related_distinct,60,0.086678,0,True
5,Q013,en,related_distinct,50,0.107317,0,True
6,Q014,en,related_distinct,55,0.092558,0,False
7,Q015,en,related_distinct,66,0.135969,0,True
8,Q018,ar,related_distinct,0,0.000000,5,None
9,Q019,ar,related_distinct,0,0.000000,5,None


In [26]:
language_summary = (
    tfidf_diagnostics
    .groupby("language")
    .agg(
        queries=("query_id", "count"),
        zero_vocab_queries=(
            "known_terms_ngrams",
            lambda x: (x == 0).sum()
        ),
        zero_top1_queries=(
            "top_cosine_similarity",
            lambda x: (x == 0).sum()
        ),
        average_known_terms=(
            "known_terms_ngrams",
            "mean"
        ),
        average_top1_similarity=(
            "top_cosine_similarity",
            "mean"
        )
    )
    .reset_index()
)

display(language_summary)

,language,queries,zero_vocab_queries,zero_top1_queries,average_known_terms,average_top1_similarity
0,ar,20,19,19,0.05,0.003524
1,en,20,0,0,52.30,0.092851


## 8. Save Development Retrieval Results

In [27]:
# Save TF-IDF development retrieval results

tfidf_export = tfidf_dev_results.drop(
    columns=["description"],
    errors="ignore"
)

tfidf_export.to_csv(
    OUTPUT_DIR / "tfidf_dev_top5.csv",
    index=False,
    encoding="utf-8-sig"
)

tfidf_diagnostics.to_csv(
    OUTPUT_DIR / "tfidf_dev_diagnostics.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(OUTPUT_DIR / "tfidf_dev_top5.csv")
print(OUTPUT_DIR / "tfidf_dev_diagnostics.csv")

Saved:
results/tfidf_dev_top5.csv
results/tfidf_dev_diagnostics.csv
